# Object Detection using SSD and Yolo
# **SSD:**

In [3]:
%pip install --upgrade numpy

Note: you may need to restart the kernel to use updated packages.


In [6]:
!pip install --upgrade pip
!pip install --upgrade numpy h5py tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 468.6 kB/s eta 0:00:00a 0:00:01
Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: h5py
    Found existing installation: h5py 3.11.0
    Uninstalling h5py-3.11.0:
      Successfully uninstalled h5py-3.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires

In [10]:
!pip install numpy==1.26.0
!pip install h5py==3.11.0
!pip install tensorflow==2.13.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 2.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires h5py>=3.11.0, but you have h5py 3.8.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 1.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: h5py
    Found existing installation: h5py 3.8.0
    Uninstalling h5py-3.8.0:
      Successfully uninstalled h5py-3.8.0
  Using cached tensorflow-2.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached keras-2.13.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached tensorboard-2.13.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorflow_e

In [11]:
import numpy as np
import tensorflow as tf

print("NumPy version:", np.__version__)
print("TensorFlow version:", tf.__version__)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# Step 1-Backbone 

In [ ]:
def build_backbone(input_shape = (300 , 300, 3)):
    inputs = tf.keras.Input(shape = input_shape)
    x = layers.Conv2D(32 , (3, 3) , activation = 'relu' , padding = 'same')(inputs)
    x = layers.MaxPooling2D((2 , 2))(x)
    x = layers.Conv2D(64 , (3,3) , activation = 'relu' , padding = 'same')(x)
    x = layers.MaxPooling2D(( 2 , 2))(x)
    x = layers.Conv2D(128 , (3,3), activation = 'relu' , padding = 'same')(x)
    base_model = models.Model(inputs , x , name = "SSD_Backbone")
    return base_model

# Step 2-SSD Prediction Head


In [ ]:
def ssd_head(x , num_classes , num_boxes):
    class_head = layers.Conv2D(num_boxes * num_classes , ( 3 , 3 ) , padding = "same")(x)
    class_head = layers.Reshape((-1 , 4))(box_head)
    
    return class_head, box_head

# Step 3-Multi-scale Feature Maps

In [ ]:
def build_ssd_model(input_shape = (300 , 300 , 3), num_classes = 21 , num_boxes = 6):
    # backbone network
    base = build_backbone(input_shape)
    x = base.output # starting feature map from the backbone
    
    # Multi-scale feature maps for detection heads
    class_outputs , box_outputs = [] , []
    for i in range(3): # 3 scales for simplicity
        class_head , box_head = ssd_head(x , num_classes, num_boxes)
        class_outputs.append(class_head)
        box_outputs.append(box_head)
        
        # Downsample the feature map from next scale
        x = layers.Conv2D(128 , (1,1) , strides = 2 , padding = "same")(x)
        
    # concatenate outputs across all scale
    classes_concat = layers.Concatenate(axis = 1)(class_outputs)
    boxes_concat = layers.Concatenate(axis = 1)(box_outputs)
    
    # Define the final SSD model
    ssd_model = models.Model(inputs = base.input , output = [classes_concat , boxes_concat])
    return ssd_model

# Instantiate SSD model
ssd_model = build_ssd_model()
ssd_model.summary()


# Step 4-Generate Dummy Data for Testing

In [ ]:
# Generaterandom dummy data and test the model
ssd_input = np.random.random((1 , 300 , 300 , 3))
ssd_class_pred , ssd_box_pred = ssd_model.predict(ssd_input)

# Print shapes of predictions to confirm the output
print("\nSSD Class Prediction Shape:", ssd_class_pred.shape)
print("SSD Box Prediction Shape:" , ssd_box_pred.shape)


# YOLO:


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# Step 1-Define the Backbone(Feature Extractor)

In [ ]:
def build_yolo_backbone(input_shape = (416 , 416 , 3)):
    inputs = tf.keras.Input(shape = input_shape)
    x = layers.Conv2D(32 , (3,3), strides = 1, padding = "same" , activation = "relu")(inputs)
    x = layers.MaxPooling2D((2,2))(x)
    
    x = layers.Conv2D(64 , (3,3), stride = 1 , padding = "same" , activation = "relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    
    x = layers.Conv2D(128 , (3,3), stride = 1 , padding = "same" , activation = "relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    
    x = layers.Conv2D(256 , (3,3), stride = 1 , padding = "same" , activation = "relu")(x)
    x = layers.MaxPooling2D((2,2))(x)
    
    x = layers.Conv2D(512 , (3,3), stride = 1 , padding = "same" , activation = "relu")(x)
    x = layers.MaxPooling2D((2,2))(x)

    return models.Model(inputs, x , name = "Yolo_Backbone")

# Step 2-Define YOLO Head(Detection Layer)

In [ ]:
def yolo_head(x , num_classes , num_boxes):
    # number of output channes: num_boxes * (5+ num_classes)
    # 5 = 4 bounding box xoordinates (x,y,w,h) + 1 confidence score
    num_outputs = num_boxes * (num_classes + 5)
    yolo_output = layers.Conv2D(num_outputs , (1,1) , padding = "same")(x)
    yolo_output = layers.Reshape((13,13, num_boxes , num_classes + 5))(yolo_output) 
    return yolo_output

# Step 3-combine Backbone and YOLO Head to build YOLO Model

In [ ]:
def build_yolo_model(input_shape = (416 , 416 , 3), num_classes = 20 , num_boxes = 3):
    backbone = build_yolo_backbone(input_shape)
    x = backbone.output
    
    # YOLO head for detection
    yolo_output = yolo_head(x , num_classes , num_boxes)
    
    # Define the YOLO model
    yolo_model = models.Model(inputs = backbone.input , outputs = yolo_output)
    return yolo_model

# instantiate YOLO model
yolo_model = build_yolo_model()
yolo_model.summary()

# Step 4-Generate Dummy Data for Testing

In [ ]:
# Generate a random dummy input image of shape(1 , 416 , )
dummy_input = np.random.random((1,416,416,3))

# Run a prediction on the dummy input
yolo_pred = yolo_model.predict(dummy_input)

# Print the shape of the prediction
print("\nYOLO Prediction Shape:" , yolo_pred.shape)